# L29 · 常见概率分布（Probability Distribution） — 均匀（Uniform）、正态（Gaussian）、伯努利（Bernoulli）：PDF / CDF 与采样（Sampling）

**目标**：实现 `gaussian_pdf`，验证正态分布的 68–95–99.7% 经验法则，用 `rng.binomial` 采样 Bernoulli 分布，用梯形数值积分（`np.trapezoid`）计算 CDF。

**为什么对 Aurora 重要**：正态分布是 He 权重初始化（`np.random.normal(0, scale, shape)`）的理论基础；Bernoulli 分布是 dropout 的概率模型（每个神经元以概率 p 独立"关闭"）；CDF 对应特征分布的分位数分析，常用于训练数据质量诊断。

← **上一课**　[L28 · 均值方差标准化](L28_descriptive_stats.ipynb)

> 上节课学习了 **均值方差标准化**：描述性统计、z-score 标准化与分布比较。  
> 本课将探讨 **常见概率分布**。

## 本课剧情：神经网络权重为什么初始化成正态分布？

你有没有想过：训练神经网络时，权重初始值从哪来？

答案是**正态分布（Gaussian distribution）**：`He 初始化`用 N(0, √(2/n)) 初始化权重，确保信号在层间传播时不会爆炸也不会消失。这个选择背后的数学基础就是本课的核心——概率分布的形状与性质。

三种最常用的分布：

| 分布 | 参数 | 典型用途 |
|---|---|---|
| **均匀分布 Uniform(a,b)** | 下界 a，上界 b | 随机 seed、数据增强随机偏移 |
| **正态分布 Normal(μ,σ)** | 均值 μ，标准差 σ | 权重初始化、噪声模型、误差分布 |
| **伯努利分布 Bernoulli(p)** | 成功概率 p | Dropout mask（每个神经元以概率 p 保留） |

本课实现 `gaussian_pdf(x, mu, sigma)`——正态分布的概率密度函数（PDF），并通过数值积分验证：PDF 在 ±∞ 上积分为 1。

## 实验入口：采样如何收敛到分布形状

生成 N=10 和 N=10000 个样本（Sample），观察直方图的轮廓：小样本时形状粗糙，大样本时逼近理论 PDF 曲线。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
u = rng.uniform(-3, 3, 5000)    # 均匀: 区间内处处等可能
g = rng.normal(0, 1, 5000)      # 正态: 中间多、两边少
fig, ax = plt.subplots(1, 2, figsize=(9,3))
ax[0].hist(u, bins=40); ax[0].set_title('uniform')
ax[1].hist(g, bins=40); ax[1].set_title('normal'); plt.show()

## 动手观察：随机代码要多试几次

多次运行下面的格，观察小样本（N=10）的均值（Mean）每次波动多大，大样本（N=10000）的均值有多稳定。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    u = rng.uniform(0, 1, size=n)
    print(f'n={n:6d}  均匀分布样本均值={np.mean(u):.3f}  '  # 理论值 0.5
          f'std={np.std(u):.3f}')           # 理论值 1/sqrt(12)≈0.289


## 代码实验：多次重复同一个随机实验

这个实验展示样本量越大、重复实验之间均值的方差越小——这是大数定律（Law of Large Numbers，LLN）的直接体现。

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        samples = rng.normal(0, 1, size=n)  # 标准正态
        estimates.append(np.mean(samples))
    print(f'n={n:4d} -> 样本均值 5次:', np.round(estimates, 3),
          '平均=', round(float(np.mean(estimates)), 3))


## 2. 正态分布（Gaussian）：最重要的连续分布

**为什么正态分布无处不在？**

中心极限定理（CLT）：足够多独立随机变量的平均值趋向正态分布——无论原始分布是什么形状。音频噪声、测量误差、神经网络梯度……都近似正态。

**概率密度函数（PDF）**：

$$f(x;\mu,\sigma) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**关键性质**：
- **峰值**：在 x=μ 处，peak = 1/(σ√2π) ≈ 0.3989（当 σ=1 时）
- **宽度**：σ 越大曲线越宽、越矮；面积始终为 1
- **68-95-99.7 规则**：μ±1σ 包含 68.27%，μ±2σ 包含 95.45%，μ±3σ 包含 99.73%

**手算例子**（标准正态 μ=0, σ=1，x=0）：
```
f(0) = 1/(1·√(2π)) = 1/√(6.283...) ≈ 0.3989
```

σ 翻倍（σ=2）时，峰值减半：f(0) = 1/(2√2π) ≈ 0.1995——但曲线更宽，面积仍为 1。

> **实现提示**：`np.exp(-((x-mu)/sigma)**2 / 2) / (sigma * np.sqrt(2*np.pi))`

写 `gaussian_pdf` 前明确三件事：
- 输入：`x`（评估点，数组），`mu`（均值），`sigma`（标准差（Standard Deviation，SD），须 > 0）
- 关键步骤：计算指数项 `exp(-(x-mu)**2 / (2*sigma**2))`，再除以归一化系数 `sigma * sqrt(2π)`
- 返回：与 `x` 形状相同的 PDF 值数组，所有值均非负

In [ ]:
def gaussian_pdf(x, mu=0.0, sigma=1.0):
    # ✏️ TODO: 返回正态分布在 x 处的概率密度
    raise NotImplementedError("请实现 gaussian_pdf — 见上方推理路线")


In [ ]:
import numpy as np

try:
    peak = gaussian_pdf(np.array([0.0]))
except NotImplementedError:
    print("⬜ 请先实现 gaussian_pdf，再运行此格")
else:
    peak_val = peak[0]
    assert abs(peak_val - 1/np.sqrt(2*np.pi)) < 1e-9, "峰值应在 μ 处"
    assert abs(gaussian_pdf(np.array([1.0]))[0] - gaussian_pdf(np.array([-1.0]))[0]) < 1e-12, "应关于 μ 对称"
    # 检查非标准参数 mu=2, sigma=0.5
    y2 = gaussian_pdf(np.array([2.0]), mu=2.0, sigma=0.5)
    assert abs(y2[0] - 1/(0.5*np.sqrt(2*np.pi))) < 1e-9, "mu=2, sigma=0.5 峰值错误"
    # 检查归一化：sigma=2 时积分应≈1
    xs_dense = np.linspace(-10, 10, 4000)
    integral = np.trapz(gaussian_pdf(xs_dense, 0.0, 2.0), xs_dense)
    assert abs(integral - 1.0) < 0.001, "sigma=2 时积分应≈1"
    print("✅ 通过：高斯密度峰值=", round(peak_val, 4))


**🔗 Aurora 连接**：`gaussian_pdf` 是 Aurora 生成流水线的密度评估原语。He 权重初始化（`np.random.normal(0, std, shape)`）、VAE 隐变量采样（`z ~ N(0, I)`）、扩散模型加噪（逐步叠加 `N(0, β_t·I)` 噪声）——三处都直接对应今天手写的公式。


In [ ]:
import numpy as np

# 固定 mu=0，改变 sigma，观察峰值和宽度的变化
xs = np.linspace(-4, 4, 100)
for sigma in [0.5, 1.0, 2.0]:
    try:
        vals = gaussian_pdf(xs, mu=0.0, sigma=sigma)
    except NotImplementedError:
        print("⬜ 请先实现 gaussian_pdf")
        break
    print(f'sigma={sigma:.1f}  峰值={gaussian_pdf(np.array([0.0]), 0.0, sigma)[0]:.4f}')
    print(f'       曲线下面积≈{np.trapz(vals, xs):.4f}  （应≈1.0）')


## 参数实验：只改一个旋钮

固定 `mu=0`，把 `sigma` 从 0.5 改到 2，观察峰值从 0.8 降到 0.2（更扁平），但曲线下面积始终为 1。这说明 sigma 控制"集中程度"，不改变总概率。再把 `mu` 从 0 改到 2，曲线整体平移但形状（峰值、宽度）不变——mu 只决定分布的中心位置。

In [ ]:
import numpy as np

# 正态分布的 CDF：用数值积分近似
xs = np.linspace(-4, 4, 1000)
for sigma in [0.5, 1.0, 2.0]:
    try:
        pdf = gaussian_pdf(xs, 0.0, sigma)
    except NotImplementedError:
        print("⬜ 请先实现 gaussian_pdf")
        break
    cdf = np.cumsum(pdf) * (xs[1] - xs[0])
    # 经验法则：μ±1σ 覆盖 ≈68% 的面积
    mask = np.abs(xs) <= sigma
    area_1sigma = np.trapz(pdf[mask], xs[mask])
    print(f'sigma={sigma:.1f}  μ±1σ 面积={area_1sigma:.3f}  (期望≈0.683)')


## 3. 伯努利分布（Bernoulli Distribution）

单次二值试验：以概率 `p` 成功（X=1），概率 `1-p` 失败（X=0）。

**PMF（概率质量函数，Probability Mass Function）**：
```
P(X=k) = p^k * (1-p)^(1-k),  k ∈ {0,1}
```

**期望（Expectation）** = `p`，**方差（Variance）** = `p(1-p)`。

**Aurora 连接**：dropout 掩码的每个元素服从 Bernoulli(p=1-dropout_rate)，独立地决定是否保留该神经元。音频增强（data augmentation）中的 SpecAugment 也用 Bernoulli 掩码随机遮挡频率段。

In [ ]:
import numpy as np

# 伯努利 PMF
def bernoulli_pmf(k, p):
    k = np.asarray(k)
    return (p ** k) * ((1 - p) ** (1 - k))

# 验证 PMF：k=0 和 k=1 的概率之和为 1
for p in [0.3, 0.5, 0.8]:
    pmf0 = bernoulli_pmf(0, p)
    pmf1 = bernoulli_pmf(1, p)
    print(f'p={p}: P(X=0)={pmf0:.3f}, P(X=1)={pmf1:.3f}, sum={pmf0+pmf1:.3f}')
    assert abs(pmf0 + pmf1 - 1.0) < 1e-9

# 采样验证：大样本均值趋近 p
rng = np.random.default_rng(0)
for p in [0.3, 0.5, 0.8]:
    samples = rng.binomial(1, p, size=100_000)
    print(f'p={p}: 样本均值={samples.mean():.4f}  方差={samples.var():.4f}  理论方差={p*(1-p):.4f}')

print('✅ 伯努利分布 PMF 与采样均值验证通过')


## 4. 累积分布函数（Cumulative Distribution Function，CDF）

CDF 定义：`F(x) = P(X ≤ x)` — 随机变量取值不超过 x 的概率。

| 分布 | CDF 形式 |
|---|---|
| 均匀 U[a,b] | F(x) = (x-a)/(b-a) 对 x∈[a,b] |
| 正态 N(μ,σ²) | 用数值积分：∫ pdf(t)dt |
| 伯努利 B(p) | F(0)=1-p，F(1)=1 |

**性质**：CDF 单调非降，F(-∞)=0，F(+∞)=1；连续分布的 CDF 处处连续。

**用途**：给定 CDF，对任意区间 [a,b] 的概率直接相减：`P(a < X ≤ b) = F(b) - F(a)`。

In [ ]:
import numpy as np

# 正态分布 CDF（数值积分：梯形法则）
def normal_cdf(x_eval, mu=0.0, sigma=1.0, n=2000):
    xs = np.linspace(mu - 6*sigma, x_eval, n)
    pdf = np.exp(-0.5*((xs - mu)/sigma)**2) / (sigma * np.sqrt(2*np.pi))
    return np.trapz(pdf, xs)

# 验证经验法则：N(0,1) 在 [-1,1] 内概率 ≈ 68.27%
p_within_1sd = normal_cdf(1.0) - normal_cdf(-1.0)
p_within_2sd = normal_cdf(2.0) - normal_cdf(-2.0)
p_within_3sd = normal_cdf(3.0) - normal_cdf(-3.0)

print(f'P(-1 < X < 1) = {p_within_1sd:.4f}  (理论 68.27%)')
print(f'P(-2 < X < 2) = {p_within_2sd:.4f}  (理论 95.45%)')
print(f'P(-3 < X < 3) = {p_within_3sd:.4f}  (理论 99.73%)')

assert abs(p_within_1sd - 0.6827) < 0.002
assert abs(p_within_2sd - 0.9545) < 0.002
assert abs(p_within_3sd - 0.9973) < 0.002
print("✅ 正态分布 CDF 经验法则验证通过")


## 本课收束

现在可以用 `gaussian_pdf(x, mu, sigma)` 在任意点求正态分布的密度值，并通过 `np.random.uniform` / `np.random.normal` 生成两种分布的样本。`gaussian_pdf` 对应 Aurora 中加噪步骤的密度评估和 VAE 的 KL 散度计算。下一节用 softmax 把未归一化得分变成概率分布，今天手写的归一化系数思路在那里会再出现。

## ✏️ 白板挑战：概率分布手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：标准正态分布（μ=0, σ=1），峰值 f(0) = ?  
（公式：1/(σ√2π)）

**问 2**：σ 从 1 增大到 2 时，峰值变为 ?（与问1的比值是多少？）

**问 3**：伯努利分布（p=0.3）：
- P(X=1) = ?
- E[X] = ?
- Var[X] = p(1-p) = ?

**问 4**：正态分布 68-95-99.7 规则：μ±2σ 覆盖大约 ?% 的面积？  
Aurora 中，若 MFCC 特征均值=0、标准差=1，那么绝大多数（>99%）特征值落在什么范围内？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：f(0; μ=0, σ=1) = 1/√(2π) ≈ 0.3989
peak_expected = 1 / np.sqrt(2 * np.pi)
assert np.isclose(peak_expected, 0.3989422804014327, atol=1e-10)
try:
    peak_computed = gaussian_pdf(np.array([0.0]), mu=0.0, sigma=1.0)[0]
    assert np.isclose(peak_computed, peak_expected, atol=1e-10)
    print(f"Q1 ✅  f(0;0,1)=1/√2π={peak_expected:.6f}，函数计算={peak_computed:.6f}")
except NotImplementedError:
    print(f"Q1 ✅  f(0;0,1)=1/√(2π)={peak_expected:.6f}  (gaussian_pdf 待实现)")

# 问2：σ=2 时峰值减半
peak_sigma2 = 1 / (2 * np.sqrt(2 * np.pi))
ratio = peak_sigma2 / peak_expected
assert np.isclose(ratio, 0.5, atol=1e-12)
print(f"Q2 ✅  σ=2时峰值={peak_sigma2:.6f}，比σ=1时小{(1/ratio):.0f}倍（面积仍=1）")

# 问3：Bernoulli(p=0.3)
p = 0.3
assert np.isclose(p, 0.3)
e_x = p             # E[X] = p
var_x = p * (1-p)   # Var[X] = p(1-p)
assert np.isclose(e_x, 0.3) and np.isclose(var_x, 0.21)
print(f"Q3 ✅  Bernoulli(0.3): P(X=1)={p}, E[X]={e_x}, Var[X]={var_x}")

# 问4：68-95-99.7规则
rng = np.random.default_rng(0)
samples = rng.normal(0, 1, 1_000_000)
within_2sigma = np.mean(np.abs(samples) <= 2)
assert abs(within_2sigma - 0.9545) < 0.005, f"μ±2σ覆盖率应≈95.45%，得到{within_2sigma:.4f}"
within_3sigma = np.mean(np.abs(samples) <= 3)
assert within_3sigma > 0.99
print(f"Q4 ✅  μ±2σ覆盖={within_2sigma:.4f}≈95.45%，μ±3σ覆盖={within_3sigma:.4f}>99%")
print("     MFCC特征(μ=0,σ=1)：>99%的值落在[-3,3]区间内")
print("\n🎉 概率分布白板挑战通过！正态/均匀/伯努利三大分布已内化。")

In [ ]:
# ✏️ 本课自评
l29_review = {
    "gaussian_pdf_formula":       None,  # 记住正态PDF公式？True/False
    "gaussian_pdf_implemented":   None,  # gaussian_pdf 实现并通过断言？True/False
    "sigma_peak_relationship":    None,  # 理解σ↑→峰值↓但面积=1？True/False
    "rule_68_95_997":             None,  # 记住68-95-99.7规则？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l29_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l29_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L29 全部通关！进入 L30：Softmax 与交叉熵')

In [ ]:
# 小检查：大样本采样的 μ±1σ 覆盖率（直接采样，无需 gaussian_pdf）
# 正态分布 μ±1σ 覆盖约 68.27% 的面积
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.normal(0, 1, size=n)
    within_1std = np.sum(np.abs(samples) <= 1.0) / n
    print(f'n={n:4d} -> μ±1σ 内的样本比例={within_1std:.3f}  (理论≈0.683)')


---

→ **下一课**　[L30 · Softmax 与交叉熵](L30_softmax_crossentropy.ipynb)

> 下节课将学习 **Softmax 与交叉熵**：分类模型的输出层与损失函数，手推梯度。